# Perceived impact of climate change

`cc4` variants ask participants to assess how much global warming/climate change is harming particular communities or locations. A positive response ($> 1$) indicates that a participant harbours some belief that climate change is happening. A negative response ($0$) could indicate belief that climate change is not happening, or that it is not impacting the specified group. The more negative responses, the more likely that a participant 
believes climate change is not happening.

In [ ]:
from pathlib import Path

import polars as pl
import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

from climate_attitudes.extract.dataset import ClimateAttitudesDataset

ASSETS_DIR = Path("/Users/henry/data/msc_thesis/climate-attitudes")

data = ClimateAttitudesDataset(ASSETS_DIR)

In [ ]:
cc4_responses = (
    data.question_response.filter(
        pl.col("item_name").is_in(
            [
                "cc4_world",
                "cc4_wealthcoun",
                "cc4_poorcoun",
                "cc4_wealthUS",
                "cc4_poorUS",
                "cc4_comm",
                "cc4_person",
            ]
        ),
        pl.col("wave") == 1,
    )
    .with_columns(pl.col("response").cast(pl.Int64).replace(99, None))
    .select("participant_id", "item_name", "response")
    .pivot("item_name", values="response")
)

In [ ]:
sns.pairplot(
    cc4_responses.drop("participant_id").to_pandas(),
    kind="hist",
    plot_kws=dict(discrete=True),
)

In [ ]:
corr = cc4_responses.drop("participant_id").to_pandas().corr()

# Generate a mask for the upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

# Set up the matplotlib figure
f, axe = plt.subplots(figsize=(11, 9))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(
    corr,
    mask=mask,
    cmap=cmap,
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.5},
)